# 01 · Data Cleaning & Feature Engineering

**Goal:** check the quality of the raw house-sales data, decide how to handle each issue, and create a clean dataset for EDA and modelling.

All logic lives in [`src/data_prep.py`](../src/data_prep.py). This notebook calls those functions and explains the decisions.

**Ground rules**
- The raw CSV is never modified.
- No rows are dropped silently. Every decision is logged.
- No data leakage: only fixed row-by-row rules here. Anything that learns from the data (median imputation, scaling) is fitted later on the **training set only**.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Make `src` importable whether the notebook runs from notebooks/ or the project root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src import config, data_prep

pd.set_option("display.max_columns", 30)

## 1. Load and validate the raw data

In [2]:
raw = data_prep.load_raw_data()
print("Raw shape:", raw.shape)

df = data_prep.drop_index_column(raw)   # remove the 'Unnamed: 0' index column
data_prep.validate_raw_schema(df)       # raises an error if columns or types are wrong
df = data_prep.parse_sale_date(df)      # '20141013T000000' -> 2014-10-13
print("Schema OK. Sale dates:", df["sale_date"].min().date(), "to", df["sale_date"].max().date())
df.head(3)

Raw shape: (21613, 22)
Schema OK. Sale dates: 2014-05-02 to 2015-05-27


,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,sale_date
0,7129300520,20141013T000000,221900.0,3.0,1.00,1180,5650,1.0,0,0,3,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650,2014-10-13
1,6414100192,20141209T000000,538000.0,3.0,2.25,2570,7242,2.0,0,0,3,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639,2014-12-09
2,5631500400,20150225T000000,180000.0,2.0,1.00,770,10000,1.0,0,0,3,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062,2015-02-25


## 2. Missing values

In [3]:
print(data_prep.summarise_missing_values(df.drop(columns="sale_date")))
df[df["bedrooms"].isna() | df["bathrooms"].isna()][
    ["id", "price", "bedrooms", "bathrooms", "sqft_living", "grade"]
]

{'bedrooms': 13, 'bathrooms': 10}


,id,price,bedrooms,bathrooms,sqft_living,grade
875,6306400140,1095000.0,NaN,NaN,3064,7
1149,3421079032,75000.0,1.0,NaN,670,3
3119,3918400017,380000.0,NaN,NaN,1470,8
3467,1453602309,288000.0,NaN,1.50,1430,7
4868,6896300380,228000.0,NaN,1.00,390,4
5832,5702500050,280000.0,1.0,NaN,600,3
6994,2954400190,1295650.0,NaN,NaN,4810,12
8477,2569500210,339950.0,NaN,2.50,2290,8
8484,2310060040,240000.0,NaN,2.50,1810,7
9773,3374500520,355000.0,NaN,NaN,2460,8


**Decision:** keep these rows and leave the values as `NaN` for now.

- Several are clearly real, normal homes (e.g. a 4,810 sq ft, grade-12 home with no bedroom count), so the value is *missing*, not zero.
- Filling with the median here would compute the median on the **whole dataset**, including future test rows. That is leakage. Instead, a `SimpleImputer(strategy="median")` inside the model pipeline will learn the median from the training data only.

## 3. Repeated IDs: duplicates or repeat sales?

In [4]:
data_prep.investigate_repeat_sales(df)

{'ids_sold_more_than_once': 176,
 'rows_with_repeated_id': 353,
 'extra_rows_beyond_first_sale': 177,
 'max_sales_per_id': 3,
 'same_id_and_same_date': 0,
 'ids_with_changed_house_attributes': 0,
 'ids_with_different_prices': 173,
 'conclusion': 'repeat sales - kept'}

In [5]:
# Example: one house, two sale dates, two different prices
repeat_ids = df.loc[df["id"].duplicated(), "id"].head(2)
df[df["id"].isin(repeat_ids)][["id", "sale_date", "price", "bedrooms", "sqft_living", "grade"]]

,id,sale_date,price,bedrooms,sqft_living,grade
93,6021501535,2014-07-25,430000.0,3.0,1580,8
94,6021501535,2014-12-23,700000.0,3.0,1580,8
313,4139480200,2014-06-18,1384000.0,4.0,4290,11
314,4139480200,2014-12-09,1400000.0,4.0,4290,11


**Decision:** keep all of them. No id appears twice on the same date, and the house attributes never change between sales. These are the **same home sold again**, which is valid market data.

*Note for modelling:* a house sold twice could land in both train and test. We will keep this in mind when evaluating.

## 4. Suspicious values

In [6]:
data_prep.find_suspicious_values(df)

{'non_positive_price': 0,
 'non_positive_sqft_living': 0,
 'bedrooms_over_10': 2,
 'sqft_per_bedroom_below_100': 1,
 'built_after_sale_year': 12,
 'renovated_after_sale_year': 6,
 'area_mismatch_above_plus_basement': 0,
 'price_above_q3_plus_3iqr': 420,
 'price_q3_plus_3iqr_cutoff': 1614150.0}

### 4a. Extreme bedroom counts

In [7]:
df["sqft_per_bedroom"] = df["sqft_living"] / df["bedrooms"]
df[df["bedrooms"] >= 10][["id", "price", "bedrooms", "bathrooms", "sqft_living", "sqft_per_bedroom", "grade"]] \
    .sort_values("bedrooms", ascending=False)

,id,price,bedrooms,bathrooms,sqft_living,sqft_per_bedroom,grade
15870,2402100895,640000.0,33.0,1.75,1620,49.090909,7
8757,1773100755,520000.0,11.0,3.00,3000,272.727273,7
13314,627300145,1148000.0,10.0,5.25,4590,459.000000,9
15161,5566100170,650000.0,10.0,2.00,3610,361.000000,7
19254,8812401450,660000.0,10.0,3.00,2920,292.000000,7


**Decision:**
- **33 bedrooms in 1,620 sq ft** is about 49 sq ft per bedroom, which is physically implausible. It is almost certainly a typo, but we don't *guess* "3". We set `bedrooms` to `NaN` and keep the row, so it gets imputed like any other missing value.
- The 10- and 11-bedroom homes have 270+ sq ft per bedroom. Unusual but plausible, so we keep them unchanged.
- Rule used: `sqft_living / bedrooms < 100` → invalid. In this dataset it flags exactly one row.

### 4b. Dates that don't line up

In [8]:
sale_year = df["sale_date"].dt.year
print("Sold before the build year (pre-construction):")
display(df[df["yr_built"] > sale_year][["id", "sale_date", "yr_built"]].head())
print("Renovation year is AFTER the sale:")
display(df[(df["yr_renovated"] > 0) & (df["yr_renovated"] > sale_year)][["id", "sale_date", "yr_renovated"]])

Sold before the build year (pre-construction):


,id,sale_date,yr_built
1763,1832100030,2014-06-25,2015
2687,3076500830,2014-10-29,2015
7526,9520900210,2014-12-31,2015
8039,1250200495,2014-06-24,2015
14489,2770601530,2014-08-26,2015


Renovation year is AFTER the sale:


,id,sale_date,yr_renovated
2295,8712100320,2014-07-28,2015
7097,9141100005,2014-10-28,2015
11599,7284900030,2014-05-22,2015
14859,3585900665,2014-06-06,2015
15687,3585900190,2014-10-06,2015
18575,8935100100,2014-07-01,2015


**Decisions:**
- Pre-construction sales would give an age of −1, so `house_age` is set to **0** (a new home).
- A renovation after the sale did not exist when the price was agreed. Using it would leak **future information**, so these rows count as **not renovated**.

### 4c. Redundant columns

In [9]:
print("sqft_above + sqft_basement == sqft_living for every row:",
      (df["sqft_above"] + df["sqft_basement"] == df["sqft_living"]).all())

sqft_above + sqft_basement == sqft_living for every row: True


`sqft_above` is a perfect linear combination of two other columns, so it adds no information and causes multicollinearity in linear models. **Dropped.**
Similarly, `house_age = sale_year − yr_built`, so `yr_built` is dropped once `house_age` exists.

## 5. Run the full cleaning pipeline

In [10]:
clean, report = data_prep.clean_data(raw)
print("Clean shape:", clean.shape)
report["decisions"]

Clean shape: (21613, 24)


{'rows_excluded': 0,
 'rows_excluded_reason': 'none - suspicious values are fixed or flagged, never silently dropped',
 'bedrooms_set_to_missing_ids': [2402100895],
 'bedrooms_set_to_missing_reason': 'sqft_living / bedrooms < 100 (physically implausible)',
 'missing_value_strategy': 'left as NaN here; impute with the training-set median inside the model pipeline',
 'repeat_sales': 'kept - same house sold on different dates is valid market data',
 'columns_dropped': {'date': 'replaced by the parsed `sale_date` column',
  'sqft_above': 'always equals sqft_living - sqft_basement (perfect linear combination)',
  'yr_built': 'replaced by `house_age` (= sale_year - yr_built), keeping both is a linear combination',
  'yr_renovated': "replaced by `is_renovated` and `years_since_update` (0 means 'never', not year 0)"},
 'features_created': ['sale_date',
  'sale_year',
  'sale_month',
  'house_age',
  'is_renovated',
  'years_since_update',
  'has_basement']}

In [11]:
report["after_cleaning"]

{'n_rows': 21613,
 'n_columns': 24,
 'missing_values': {'bedrooms': 14, 'bathrooms': 10},
 'pre_construction_sales_age_set_to_0': 12,
 'renovations_after_sale_ignored': 6}

## 6. New features

In [12]:
new_features = ["sale_year", "sale_month", "house_age", "is_renovated", "years_since_update", "has_basement"]
clean[new_features].describe().T[["mean", "min", "max"]].round(2)

,mean,min,max
sale_year,2014.32,2014.0,2015.0
sale_month,6.57,1.0,12.0
house_age,43.32,0.0,115.0
is_renovated,0.04,0.0,1.0
years_since_update,40.96,0.0,115.0
has_basement,0.39,0.0,1.0


| Feature | Definition |
|---|---|
| `sale_year`, `sale_month` | From the parsed sale date (captures time/seasonality) |
| `house_age` | `sale_year − yr_built`, minimum 0 |
| `is_renovated` | 1 if renovated **on or before** the sale year |
| `years_since_update` | Years since build or last renovation, whichever is more recent |
| `has_basement` | 1 if `sqft_basement > 0` |

**Not created on purpose:** `price_per_sqft`. It is computed from the target (price), so feeding it to the model would leak the answer.

## 7. Save

In [13]:
data_prep.save_outputs(clean, report)
print("Saved:", config.CLEAN_DATA_PATH.relative_to(config.PROJECT_ROOT))
print("Raw file still has", len(pd.read_csv(config.RAW_DATA_PATH)), "rows (unchanged)")

Saved: data/processed/house_sales_clean.csv
Raw file still has 21613 rows (unchanged)


## Summary

| Issue | Rows | Decision |
|---|---|---|
| Missing bedrooms / bathrooms | 13 / 10 | Kept as NaN → median-imputed in the model pipeline (fit on train only) |
| 33 bedrooms in 1,620 sq ft | 1 | Bedrooms set to NaN, row kept |
| Repeated ids | 177 extra rows (176 homes) | Repeat sales, kept |
| Built after sale year | 12 | `house_age` = 0 |
| Renovated after sale | 6 | Treated as not renovated (avoids future leakage) |
| `sqft_above`, `yr_built`, `yr_renovated`, `date` | n/a (columns) | Dropped as redundant / replaced by features |
| **Rows excluded** | **0** | |

Next: **02 · EDA & market insights**.